# 경계층(ZPG TBL) 검증 시각화 — Schlatter & Örlü (2010) 그림 형식

`stats/<case>/bl_stats_*.dat` 와 로그를 읽어 KTH(Schlatter & Örlü 2010)·Torroja(Jiménez et al. 2010) DNS 와 비교한다.
그림은 노트북 안에서만 보여 준다(파일 저장 없음).

- §1 모니터 시계열 (dt, C_f(x=Lx/2), 재순환면 u_τ·Re_θ·자유류)
- §2 적분량 vs Re_θ (논문 fig.2): c_f, H₁₂, Re_τ
- §3 평균 속도 U⁺(y⁺) (논문 fig.3)
- §4 레이놀즈 응력 rms⁺, −uw⁺ (논문 fig.4)

**케이스는 아래 `CASES` 에서 직접 지정**한다. `stats` 를 생략하면 그 폴더의 가장 최근 파일, `"accumulated"` 면 누적 시간이 가장 긴 파일을 쓴다.


In [ ]:
import glob, os, re, sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

APP_DIR = "/shared/home/wel1come1234/workspace/MPM-STD/apps/boundary_layer"
REF_DIR = os.path.join(APP_DIR, "plot", "ref")
KAPPA, B_LOG = 0.41, 5.2

# ── 그릴 케이스 (리스트 순서 = 범례 순서) ────────────────────────────────────
#   label : 그림 이름   stats : 통계 폴더   log : 로그 글롭(가장 최근 1개)   color : 생략 가능
CASES = [
    dict(label="re4500", stats="stats/re4500", log="logs/bl_re4500_*.log"),
    dict(label="re6000", stats="stats/re6000", log="logs/bl_re6000_*.log"),
]
PICK = "accumulated"   # "latest" | "accumulated" (누적 시간이 가장 긴 파일)


In [ ]:
# ── 읽기 ──────────────────────────────────────────────────────────────────
def header_value(line, key):
    m = re.search(rf"\b{key}=([-0-9.eE+]+)", line)
    return float(m.group(1)) if m else np.nan

def read_stats(path):
    lines = open(path).read().split("\n")
    head = lines[0]
    sec1, stations, cur = [], [], None
    for ln in lines[1:]:
        if ln.startswith("# section 2"):
            meta = {k: float(v) for k, v in re.findall(r"(\w+)=([-0-9.eE+]+)", ln)}
            cur = dict(meta=meta, rows=[]); stations.append(cur); continue
        if ln.startswith("#") or not ln.strip():
            continue
        vals = [float(v) for v in ln.split()]
        (sec1 if cur is None else cur["rows"]).append(vals)
    a = np.array(sec1)
    integ = dict(x=a[:, 0], cf=a[:, 2], d99=a[:, 3], dstar=a[:, 4], theta=a[:, 5], H=a[:, 6],
                 re_theta=a[:, 7], re_tau=a[:, 8], u_tau=a[:, 9],
                 u_edge=a[:, 10] if a.shape[1] > 10 else np.ones(len(a)))
    for st in stations:
        r = np.array(st["rows"])
        st["prof"] = dict(z=r[:, 0], yp=r[:, 1], U=r[:, 2], urms=r[:, 3], vrms=r[:, 4], wrms=r[:, 5], uw=r[:, 6])
    return dict(path=path, t=header_value(head, "t"), acc=header_value(head, "accumulated_time"),
                integ=integ, stations=stations)

def pick_stats(folder):
    files = glob.glob(os.path.join(APP_DIR, folder, "bl_stats_*.dat"))
    if not files:
        return None
    if PICK == "accumulated":
        acc = {p: header_value(open(p).readline(), "accumulated_time") for p in files}
        if max(acc.values()) >= 1:                       # 1 time 미만은 재시작 직후 스냅샷 — 비교 대상 아님
            return max(files, key=lambda p: (acc[p], os.path.getmtime(p)))
    return max(files, key=os.path.getmtime)

def read_log(pattern):
    files = sorted(glob.glob(os.path.join(APP_DIR, pattern)), key=os.path.getmtime)
    if not files:
        return None
    rows, rec = [], []
    for ln in open(files[-1]):
        p = ln.split()
        if len(p) == 8 and not ln.startswith("#"):
            try: rows.append([float(v) for v in p])
            except ValueError: pass
        elif ln.startswith("#   recycle:"):
            kv = dict(re.findall(r"(\w+) ([-0-9.eE+]+)", ln.replace("(Re_theta_r", "Re_theta_r").replace(")", "")))
            rec.append(kv)
    m = np.array(rows) if rows else np.zeros((0, 8))
    out = dict(path=files[-1], step=m[:, 0], t=m[:, 1], dt=m[:, 2], div=m[:, 3], cf_mid=m[:, 5])
    out["recycle"] = rec
    return out

# ── 참조 DNS ──────────────────────────────────────────────────────────────
def read_kth(path):
    h = open(path).read(); g = lambda k: float(re.search(k + r"\s*=\s*([0-9.eE+-]+)", h).group(1))
    d = np.loadtxt(path, comments="%")
    return dict(src="KTH", re_theta=g(r"Re_\{\\theta\}"), re_tau=g(r"Re_\{\\tau\}"), H=g(r"H_\{12\}"), cf=g(r"c_f"),
                yp=d[:, 1], U=d[:, 2], urms=d[:, 3], v_wall=d[:, 4], w_span=d[:, 5], uv=d[:, 6])

def read_torroja(path):
    h = open(path).read(); g = lambda k: float(re.search(k + r"\s*=\s*([0-9.eE+-]+)", h).group(1))
    ut, ds, th = g(r"u_\{tau\}"), g(r"delta\^\*"), g(r"theta ")
    d = np.loadtxt(path, comments="%")
    return dict(src="Torroja", re_theta=g(r"Re_\{theta\}"), re_tau=g(r"Re_\{tau\}"), H=ds / th, cf=2 * ut ** 2,
                yp=d[:, 1], U=d[:, 6], urms=d[:, 2], v_wall=d[:, 3], w_span=d[:, 4], uv=d[:, 5])

REFS = [read_kth(f) for f in sorted(glob.glob(os.path.join(REF_DIR, "vel_*_dns.prof")))]
REFS += [read_torroja(f) for f in sorted(glob.glob(os.path.join(REF_DIR, "torroja", "Re_theta.*.prof")))]
nearest_ref = lambda rt: min(REFS, key=lambda r: abs(r["re_theta"] - rt))

data = []
for i, spec in enumerate(CASES):
    sp = pick_stats(spec["stats"]); lg = read_log(spec["log"])
    c = dict(label=spec["label"], color=spec.get("color", plt.cm.tab10(i)),
             stats=read_stats(sp) if sp else None, log=lg)
    data.append(c)
    s = c["stats"]
    print(f"[{c['label']}] log: {os.path.basename(lg['path']) if lg else '없음'} (t {lg['t'][-1]:.1f})" if lg else f"[{c['label']}] log 없음",
          f"| stats: {os.path.basename(s['path'])} t={s['t']:.1f} 누적 {s['acc']:.1f}" if s else "| stats 없음")
print("참조:", ", ".join(f"{r['src']} {r['re_theta']:.0f}" for r in REFS))


## 1. 모니터 시계열 — dt, C_f(x = L_x/2), 재순환면 스케일

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for c in data:
    lg = c["log"]
    if lg is None or len(lg["t"]) == 0: continue
    col = c["color"]
    ax[0].plot(lg["t"], lg["dt"], color=col, lw=1, label=c["label"])
    ax[1].plot(lg["t"], lg["cf_mid"], color=col, lw=1)
    rec = lg["recycle"]
    if rec:
        tr = np.linspace(lg["t"][0], lg["t"][-1], len(rec))   # recycle 줄은 모니터 간격마다 하나
        ut = [float(r.get("u_tau_r", np.nan)) for r in rec]
        rt = [float(r.get("Re_theta_r", np.nan)) for r in rec]
        ue = [float(r.get("U_e_r", np.nan)) for r in rec]
        ax[2].plot(tr, ut, color=col, lw=1)
        ax[3].plot(tr, rt, color=col, lw=1, label=f"{c['label']} recycle plane")
        if np.isfinite(ue).any():
            a2 = ax[3].twinx(); a2.plot(tr, ue, color=col, lw=0.8, ls=":"); a2.set_ylabel("U_e,r (dotted)"); a2.set_ylim(0.9, 1.2)
ax[0].set_ylabel("dt"); ax[1].set_ylabel("C_f(x = Lx/2)  instant"); ax[2].set_ylabel("u_tau at recycle plane")
ax[3].set_ylabel("Re_theta at recycle plane")
ax[1].axhline(0.0045, color="k", ls="--", lw=0.8)
for a in ax: a.set_xlabel("t")
ax[0].legend(fontsize=8)
fig.tight_layout(); plt.show()


## 2. 적분량 vs Re_θ — 논문 fig.2 (선 = 우리 x-분포, 점 = 비교 스테이션)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
rt = np.logspace(np.log10(400), np.log10(4500), 200)
ax[0].plot(rt, 0.024 * rt ** -0.25, "k--", lw=1, label="Smits et al. 1983")
ax[0].plot(rt, 2.0 / (np.log(rt) / 0.384 + 4.127) ** 2, "k:", lw=1.2, label="Coles–Fernholz")
ax[2].plot(rt, 1.13 * rt ** 0.843, "k--", lw=1, label="S&Ö 2010: 1.13 Re_θ^0.843")
for src, mk in (("KTH", "s"), ("Torroja", "D")):
    pts = [r for r in REFS if r["src"] == src]
    for k, key in enumerate(("cf", "H", "re_tau")):
        ax[k].plot([p["re_theta"] for p in pts], [p[key] for p in pts], mk, color="k", mfc="white", ms=7,
                   label=f"{src} DNS" if k != 1 or src else None)
for c in data:
    s = c["stats"]
    if s is None: continue
    g, col = s["integ"], c["color"]; ok = g["re_theta"] > 0
    tag = f"{c['label']} (t {s['t']:.0f}, avg {s['acc']:.0f})"
    for k, key in enumerate(("cf", "H", "re_tau")):
        ax[k].plot(g["re_theta"][ok], g[key][ok], "-", color=col, lw=1.2, alpha=0.9, label=tag if k == 0 else None)
    for st in s["stations"]:
        m = st["meta"]
        for k, key in enumerate(("Cf", "H", "Re_tau")):
            ax[k].plot(m["Re_theta"], m[key], "o", color=col, ms=7)
ax[0].set_ylabel("$c_f$"); ax[0].set_ylim(0.002, 0.007)
ax[1].set_ylabel("$H_{12}$"); ax[1].set_ylim(1.3, 1.7)
ax[2].set_ylabel(r"$Re_\tau$"); ax[2].set_yscale("log")
for a in ax: a.set_xscale("log"); a.set_xlim(400, 4500); a.set_xlabel(r"$Re_\theta$")
ax[0].legend(fontsize=7); ax[2].legend(fontsize=7)
fig.tight_layout(); plt.show()

# 자유류 u_edge(x) — ZPG 품질 (1 에서 벗어나면 상단 BC 문제)
fig, a = plt.subplots(figsize=(7, 3))
for c in data:
    if c["stats"] is None: continue
    g = c["stats"]["integ"]; a.plot(g["x"], g["u_edge"], color=c["color"], label=c["label"])
a.axhline(1, color="k", lw=0.8); a.set_xlabel("x"); a.set_ylabel("u_edge(x)"); a.legend(fontsize=8)
fig.tight_layout(); plt.show()


## 3. 평균 속도 U⁺(y⁺) — 논문 fig.3 (위: 프로파일, 아래: U⁺ − U⁺_ref)

In [ ]:
stations = [(c, st) for c in data if c["stats"] for st in c["stats"]["stations"]]
yl = np.logspace(0, 3.5, 200)
if stations:
    n = len(stations)
    fig, ax = plt.subplots(2, n, figsize=(4.6 * n, 8), squeeze=False)
    for q, (c, st) in enumerate(stations):
        m, pr, col = st["meta"], st["prof"], c["color"]; r = nearest_ref(m["Re_theta"])
        a = ax[0, q]
        a.plot(r["yp"], r["U"], "k-", lw=1.6, label=f"{r['src']} Re_θ {r['re_theta']:.0f}")
        a.plot(pr["yp"], pr["U"], "-", color=col, lw=1.3, label=f"{c['label']} Re_θ {m['Re_theta']:.0f}")
        a.plot(yl, np.log(yl) / KAPPA + B_LOG, "k:", lw=0.9, label="log law (0.41, 5.2)")
        a.plot(yl, yl, "k-.", lw=0.7)
        a.set_xscale("log"); a.set_xlim(0.5, 2000); a.set_ylim(0, 27); a.set_xlabel("$y^+$"); a.set_ylabel("$U^+$")
        a.set_title(f"{c['label']}: Re_θ {m['Re_theta']:.0f}, Re_τ {m['Re_tau']:.0f}, c_f {m['Cf']:.4f}, H {m['H']:.3f}\n"
                    f"{r['src']}: Re_θ {r['re_theta']:.0f}, Re_τ {r['re_tau']:.0f}, c_f {r['cf']:.4f}, H {r['H']:.3f}", fontsize=8)
        a.legend(fontsize=7)
        b = ax[1, q]
        b.plot(pr["yp"], pr["U"] - np.interp(pr["yp"], r["yp"], r["U"]), color=col, lw=1.3)
        b.axhspan(-0.3, 0.3, color="green", alpha=0.12); b.axhline(0, color="k", lw=0.8)
        b.set_xscale("log"); b.set_xlim(0.5, 2000); b.set_ylim(-2, 2); b.set_xlabel("$y^+$"); b.set_ylabel("$U^+ - U^+_{ref}$")
    fig.tight_layout(); plt.show()
else:
    print("스테이션 프로파일 없음")


## 4. 레이놀즈 응력 — 논문 fig.4 (참조의 v=벽수직·w=스팬 → 우리 w·v 와 짝)

In [ ]:
if stations:
    n = len(stations)
    fig, ax = plt.subplots(n, 4, figsize=(17, 3.6 * n), squeeze=False)
    for q, (c, st) in enumerate(stations):
        m, pr, col = st["meta"], st["prof"], c["color"]; r = nearest_ref(m["Re_theta"])
        pairs = ((pr["urms"], r["urms"], "$u^+_{rms}$"), (pr["vrms"], r["w_span"], "$v^+_{rms}$ (span)"),
                 (pr["wrms"], r["v_wall"], "$w^+_{rms}$ (wall-normal)"), (-pr["uw"], -r["uv"], r"$-\overline{uw}^+$"))
        for p, (ours, ref, lab) in enumerate(pairs):
            a = ax[q, p]
            a.plot(r["yp"], ref, "k-", lw=1.6, label=f"{r['src']} Re_θ {r['re_theta']:.0f}")
            a.plot(pr["yp"], ours, "-", color=col, lw=1.3, label=f"{c['label']} Re_θ {m['Re_theta']:.0f}")
            a.set_xscale("log"); a.set_xlim(0.5, 2000); a.set_xlabel("$y^+$"); a.set_ylabel(lab)
            if p == 0: a.legend(fontsize=7)
    fig.tight_layout(); plt.show()

    print(f"{'case':>8s} {'Re_θ':>6s} {'ref':>12s} {'ΔU+ log RMS':>12s} {'u pk':>12s} {'v pk':>12s} {'w pk':>12s} {'-uw pk':>12s}")
    for c, st in stations:
        m, pr = st["meta"], st["prof"]; r = nearest_ref(m["Re_theta"])
        sel = (pr["yp"] > 30) & (pr["yp"] < 0.3 * m["Re_tau"])
        dU = pr["U"][sel] - np.interp(pr["yp"][sel], r["yp"], r["U"])
        rms = np.sqrt(np.mean(dU ** 2)) if sel.any() else np.nan
        pk = lambda a, b: f"{a.max():.2f}/{b.max():.2f}"
        print(f"{c['label']:>8s} {m['Re_theta']:6.0f} {r['src'] + ' ' + str(int(round(r['re_theta']))):>12s} {rms:12.3f} "
              f"{pk(pr['urms'], r['urms']):>12s} {pk(pr['vrms'], r['w_span']):>12s} {pk(pr['wrms'], r['v_wall']):>12s} {pk(-pr['uw'], -r['uv']):>12s}")
